# Étendue of the Vera C. Rubin Observatory

Étendue ($A\Omega$) quantifies a telescope's survey power — it is the product of the
effective collecting area $A_{\mathrm{eff}}$ and the solid angle $\Omega$ of the field of view:

$$A\Omega = A_{\mathrm{eff}} \times \Omega$$

This notebook walks through each step, arriving at the canonical value of
**≈ 319 m² deg²**.

In [ ]:
import numpy as np

## 1. Telescope Parameters

Rubin uses a unique three-mirror anastigmat design where the primary mirror (M1)
and the tertiary mirror (M3) are polished onto the **same monolithic glass substrate**.
This means M1 is an annulus — it doesn't extend to the centre.

In [ ]:
# Mirror dimensions (metres)
D_M1_outer = 8.4      # M1 outer diameter
D_M1_inner = 5.012    # M1 inner diameter (= M3 outer diameter)
D_M2       = 3.42     # M2 (secondary) diameter

# Field of view
fov_diameter_deg = 3.5  # degrees

# Additional obscuration fraction (camera body, spider vanes, baffles)
additional_obscuration = 0.07  # ~7 %

print(f"M1 outer diameter : {D_M1_outer} m")
print(f"M1 inner diameter : {D_M1_inner} m")
print(f"M2 diameter       : {D_M2} m")
print(f"FoV diameter      : {fov_diameter_deg}°")

## 2. Geometric Collecting Area of the M1 Annulus

Because M2 (3.42 m) is *smaller* than the M1 central hole (5.012 m), the secondary
does not create any extra obscuration — the hole in M1 already accounts for it.

$$A_{M1} = \frac{\pi}{4}\left(D_{\mathrm{outer}}^{2} - D_{\mathrm{inner}}^{2}\right)$$

In [ ]:
A_M1 = (np.pi / 4) * (D_M1_outer**2 - D_M1_inner**2)

print(f"M1 annulus area = (π/4) × ({D_M1_outer}² − {D_M1_inner}²)")
print(f"                = (π/4) × ({D_M1_outer**2:.2f} − {D_M1_inner**2:.3f})")
print(f"                = (π/4) × {D_M1_outer**2 - D_M1_inner**2:.3f}")
print(f"                = {A_M1:.2f} m²")

## 3. Effective Collecting Area

We apply a correction for the additional obscuration from the camera body,
spider vanes, and baffling (~7 %):

$$A_{\mathrm{eff}} = A_{M1} \times (1 - f_{\mathrm{obsc}})$$

In [ ]:
A_eff = A_M1 * (1 - additional_obscuration)

print(f"Additional obscuration : {additional_obscuration*100:.0f} %")
print(f"A_eff = {A_M1:.2f} × {1 - additional_obscuration:.2f} = {A_eff:.2f} m²")

## 4. Field-of-View Solid Angle

The focal plane subtends a 3.5° diameter circle on the sky.

### Flat-sky (small-angle) approximation

$$\Omega \approx \pi \left(\frac{\theta_{\mathrm{FoV}}}{2}\right)^{2} \quad [\mathrm{deg}^{2}]$$

### Exact spherical formula

$$\Omega_{\mathrm{exact}} = 2\pi\left(1 - \cos\frac{\theta_{\mathrm{FoV}}}{2}\right) \quad [\mathrm{sr}]$$

In [ ]:
# --- In square degrees (flat-sky) ---
half_angle_deg = fov_diameter_deg / 2
Omega_deg2 = np.pi * half_angle_deg**2

# --- Exact (steradians) ---
half_angle_rad = np.radians(half_angle_deg)
Omega_sr = 2 * np.pi * (1 - np.cos(half_angle_rad))

# Convert exact steradians → square degrees for comparison
Omega_sr_to_deg2 = Omega_sr * (180 / np.pi)**2

print(f"Half-angle           : {half_angle_deg}° = {half_angle_rad:.6f} rad")
print(f"Ω (flat-sky approx)  : {Omega_deg2:.4f} deg²")
print(f"Ω (exact spherical)  : {Omega_sr:.6e} sr = {Omega_sr_to_deg2:.4f} deg²")
print(f"\nDifference           : {abs(Omega_deg2 - Omega_sr_to_deg2):.4f} deg² "
      f"({abs(Omega_deg2 - Omega_sr_to_deg2)/Omega_deg2*100:.3f} %)")
print("\n→ At 3.5° the flat-sky approximation is excellent.")

## 5. Étendue Calculation

$$A\Omega = A_{\mathrm{eff}} \times \Omega$$

In [ ]:
etendue = A_eff * Omega_deg2

print("=" * 50)
print(f"  A_eff  = {A_eff:.2f} m²")
print(f"  Ω      = {Omega_deg2:.4f} deg²")
print(f"  ────────────────────────────────")
print(f"  AΩ     = {etendue:.1f} m² deg²")
print("=" * 50)

## 6. Sensitivity to Assumptions

The canonical 319 value depends on the assumed additional obscuration.
Let's see how the étendue varies.

In [ ]:
print(f"{'Obscuration (%)':>16}  {'A_eff (m²)':>11}  {'AΩ (m² deg²)':>13}")
print("-" * 46)

for obsc_pct in [0, 3, 5, 7, 10]:
    A = A_M1 * (1 - obsc_pct / 100)
    AO = A * Omega_deg2
    marker = "  ← canonical" if obsc_pct == 7 else ""
    print(f"{obsc_pct:>15.0f}%  {A:>11.2f}  {AO:>13.1f}{marker}")

## 7. Comparison with Other Survey Telescopes

We compare Rubin's étendue against a range of wide-field survey facilities.
For each telescope we use the primary mirror geometry and the optical field of
view delivered to the detector. Where the FoV is circular we specify the diameter;
for non-circular or detector-limited fields we specify the effective sky area directly.

| Key | Meaning |
|-----|--------|
| `fov_diam` | Circular FoV diameter (°) — area computed as π(d/2)² |
| `fov_area` | Effective detector FoV area (deg²), used when field is non-circular or detector-limited |

In [ ]:
# Each entry: D_outer (m), D_inner (m), obsc_extra (fraction),
#             and EITHER fov_diam (°) OR fov_area (deg²)
#
# Notes on individual telescopes:
#   CFHT/MegaCam    — 3.58 m Cassegrain, 0.96° × 0.94° mosaic
#   SDSS            — 2.5 m modified Ritchey-Chrétien, 3.0° FoV
#   SkyMapper       — 1.35 m modified Cassegrain, 5.7 deg² FoV
#   PTF (P48)       — 1.22 m Oschin Schmidt (no central obsc.), 7.26 deg² CCD mosaic
#   Pan-STARRS1     — 1.8 m Ritchey-Chrétien, 3.0° circular FoV
#   VST/OmegaCAM    — 2.61 m modified Ritchey-Chrétien, 1° × 1° FoV
#   DECam/Blanco    — 3.934 m, 2.2° diameter FoV (used for DES)
#   VISTA/VIRCAM    — 4.1 m, 1.65° diameter optical field
#   Subaru/HSC      — 8.2 m, 1.5° diameter prime-focus FoV
#   Gemini N / GMOS  — 8.1 m Cassegrain, GMOS 5.5′ × 5.5′ FoV
#   Gemini S / GMOS  — 8.1 m Cassegrain, GMOS 5.5′ × 5.5′ FoV
#   Rubin/LSST       — 8.4 m (M1 annulus), 3.5° diameter FoV

surveys = [
    # (Name,                  D_out, D_in,  obsc,  fov_diam, fov_area)
    # Use None for whichever of fov_diam / fov_area is not specified
    ("CFHT / MegaCam",       3.58,  1.40,  0.05,  None,     0.90),
    ("SDSS 2.5 m",           2.50,  1.08,  0.05,  3.0,      None),
    ("SkyMapper 1.35 m",     1.35,  0.51,  0.05,  None,     5.69),
    ("PTF / Oschin Schmidt", 1.22,  0.00,  0.02,  None,     7.26),
    ("Pan-STARRS1 1.8 m",    1.80,  0.91,  0.05,  3.0,      None),
    ("VST / OmegaCAM",       2.61,  0.94,  0.05,  None,     1.00),
    ("DECam / Blanco (DES)", 3.934, 1.40,  0.05,  2.2,      None),
    ("VISTA / VIRCAM",       4.10,  1.24,  0.05,  1.65,     None),
    ("Gemini North / GMOS",  8.10,  1.023, 0.05,  None,     (5.5/60)**2),  # 5.5′ × 5.5′
    ("Gemini South / GMOS",  8.10,  1.023, 0.05,  None,     (5.5/60)**2),  # 5.5′ × 5.5′
    ("Subaru / HSC",         8.20,  2.24,  0.05,  1.5,      None),
    ("Rubin / LSST",         8.40,  5.012, 0.07,  3.5,      None),
]

# ── Compute and display ──────────────────────────────────────────────
print(f"{'Telescope':<25} {'D_out':>5} {'D_in':>5} {'A_eff':>8} "
      f"{'Ω (deg²)':>9} {'AΩ (m²deg²)':>12}")
print("=" * 70)

results = []
for (name, D_out, D_in, obsc, fov_d, fov_a) in surveys:
    A = (np.pi / 4) * (D_out**2 - D_in**2) * (1 - obsc)
    if fov_d is not None:
        O = np.pi * (fov_d / 2)**2          # circular FoV
    else:
        O = fov_a                            # direct area
    AO = A * O
    results.append((name, D_out, D_in, A, O, AO))

# Sort by étendue
results.sort(key=lambda r: r[5])

for (name, D_out, D_in, A, O, AO) in results:
    marker = " ◀" if "Rubin" in name else ""
    print(f"{name:<25} {D_out:>5.2f} {D_in:>5.2f} {A:>7.2f}  {O:>8.2f}  {AO:>11.1f}{marker}")

### Bar chart

In [ ]:
import matplotlib.pyplot as plt

names   = [r[0] for r in results]
etendues = [r[5] for r in results]

colors = ['#2196F3' if 'Rubin' not in n else '#FF5722' for n in names]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, etendues, color=colors, edgecolor='white', linewidth=0.5)

# Annotate values
for bar, val in zip(bars, etendues):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

ax.set_xlabel('Étendue  $A\\Omega$  (m² deg²)', fontsize=12)
ax.set_title('Survey Telescope Étendue Comparison', fontsize=14, fontweight='bold')
ax.set_xlim(0, max(etendues) * 1.18)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('etendue_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → etendue_comparison.png')

## Summary

Using Rubin's M1 annulus geometry ($D_{\mathrm{outer}} = 8.4$ m, $D_{\mathrm{inner}} = 5.012$ m)
with ~7 % additional obscuration and a 3.5° circular field of view:

$$\boxed{A\Omega \approx 319 \; \mathrm{m^{2}\,deg^{2}}}$$

This is roughly an **order of magnitude** larger than any previous survey telescope.